2) Map-Reduce 요약

In [ ]:
# [목적] PyPDFLoader로 PDF 일부 페이지를 불러와 Map-Reduce 요약용 문서 목록을 준비합니다.
# PDF를 페이지별 Document 객체로 변환한 뒤 예제에서 사용할 4~8페이지만 선택합니다.
# 선택한 docs는 각 페이지를 따로 요약하는 Map 단계의 입력으로 사용합니다.
from langchain_community.document_loaders import PyPDFLoader

# PyPDFLoader는 PDF의 각 페이지를 본문과 메타데이터가 담긴 Document로 읽습니다.
loader = PyPDFLoader(
    r"D:\shproject\ex0929\data\SPRI_AI_Brief_2023년12월호_F.pdf"
)

docs = loader.load()
docs = docs[3:8]  # 4~8페이지에 해당하는 부분
print(f"총 페이지수: {len(docs)}")

In [ ]:
# [목적] Map 단계에서 각 문서를 개별 요약할 모델과 프롬프트를 준비합니다.
# ChatOpenAI 모델을 만들고 LangSmith Hub에서 Map 전용 프롬프트를 가져옵니다.
# 준비한 구성요소는 다음 셀에서 문서별 요약 체인으로 연결합니다.
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# ChatOpenAI는 OpenAI 채팅 모델을 LangChain 체인에서 사용할 수 있게 감싼 객체입니다.
llm = ChatOpenAI(
    # 0에 가까울수록 무작위성을 줄여 일관된 요약을 생성합니다.
    temperature=0,
    model_name="gpt-4o-mini"
)

client = Client()

# pull_prompt는 LangSmith Hub에 저장된 공개 프롬프트를 현재 Notebook으로 가져옵니다.
map_prompt = client.pull_prompt(
    "teddynote/map-prompt",
    dangerously_pull_public_prompt=True
)

map_prompt.pretty_print()

In [ ]:
# [목적] Map 프롬프트와 모델을 연결해 문서별 요약을 생성하는 체인을 만듭니다.
# 파이프(|) 순서에 따라 프롬프트 구성, 모델 호출, 문자열 변환이 차례로 실행됩니다.
# StrOutputParser가 모델 응답을 일반 문자열로 바꿔 Reduce 단계에서 쉽게 합칠 수 있게 합니다.
map_chain = map_prompt | llm | StrOutputParser()

In [ ]:
# [목적] Map 체인을 여러 PDF 페이지에 일괄 실행해 페이지별 요약을 만듭니다.
# batch는 docs의 각 Document에 같은 체인을 적용하고 입력 순서대로 결과를 반환합니다.
# 생성된 요약 목록은 Reduce 단계에서 하나의 최종 요약으로 통합합니다.
doc_summaries = map_chain.batch(docs)

In [ ]:
# [목적] Map 단계에서 생성된 첫 번째 페이지 요약을 미리 확인합니다.
# doc_summaries의 첫 항목을 출력해 개별 요약이 의도대로 생성되었는지 점검합니다.
# 확인한 요약들을 다음 Reduce 단계에서 하나의 내용으로 합칩니다.
print(doc_summaries[0])

In [ ]:
# [목적] 페이지별 요약을 하나로 통합하기 위한 Reduce 전용 프롬프트를 준비합니다.
# LangSmith Hub에서 공개 프롬프트를 가져온 뒤 입력 구조를 화면에 출력합니다.
# 이 프롬프트는 doc_summaries를 받아 최종 요약을 생성하는 체인에 사용합니다.
# 공개된 Reduce 프롬프트를 내려받을 수 있도록 허용합니다.
reduce_prompt = client.pull_prompt(
    "teddynote/reduce-prompt",
    dangerously_pull_public_prompt=True
)

# 프롬프트 출력
reduce_prompt.pretty_print()

In [ ]:
# [목적] Reduce 프롬프트와 모델을 연결해 개별 요약을 최종 요약으로 합치는 체인을 만듭니다.
# Reduce 프롬프트에 요약 목록을 넣고 모델을 호출한 뒤 결과를 문자열로 변환합니다.
# 이 체인은 Map 단계의 여러 결과를 한 번에 읽기 쉬운 최종 결과로 정리합니다.
reduce_chain = reduce_prompt | llm | StrOutputParser()

In [ ]:
# [목적] 페이지별 요약을 Reduce 체인에 전달해 한국어 최종 요약을 스트리밍으로 출력합니다.
# 요약 목록을 줄바꿈으로 연결하고 출력 언어를 지정한 뒤 생성 결과를 순서대로 받습니다.
# 스트리밍 응답은 완성될 때까지 기다리지 않고 생성되는 내용을 바로 확인하는 데 사용합니다.
from langchain_teddynote.messages import stream_response

# stream은 생성되는 답변 조각을 차례로 전달하는 반복 가능한 응답을 반환합니다.
answer = reduce_chain.stream(
    {"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}
)

# stream_response가 answer의 각 조각을 받아 Notebook 화면에 연속해서 표시합니다.
stream_response(answer)

In [ ]:
# [목적] Map과 Reduce 전 과정을 하나의 재사용 가능한 LangChain 체인으로 구성합니다.
# 입력 문서를 각각 요약한 뒤 그 결과를 다시 통합하고, 최종 답변은 스트리밍으로 표시합니다.
# 여러 셀에서 확인한 절차를 함수 하나로 묶어 동일한 형식의 다른 문서에도 적용할 수 있습니다.
from langchain_core.runnables import chain
from langchain_teddynote.callbacks import StreamingCallback

# @chain은 일반 Python 함수를 invoke로 실행할 수 있는 LangChain Runnable로 변환합니다.
@chain
def map_reduce_chain(docs):
    map_llm = ChatOpenAI(
        temperature=0,
        model_name="gpt-4o-mini",
    )

    # map prompt 다운로드
    map_prompt = client.pull_prompt(
        "teddynote/map-prompt",
        dangerously_pull_public_prompt=True
    )

    # Map 프롬프트 → 모델 → 문자열 변환 순서로 문서별 요약 체인을 구성합니다.
    map_chain = map_prompt | map_llm | StrOutputParser()

    # batch로 각 문서를 개별 요약해 Reduce 단계에 전달할 목록을 만듭니다.
    doc_summaries = map_chain.batch(docs)

    # reduce prompt 다운로드
    reduce_prompt = client.pull_prompt(
        "teddynote/reduce-prompt",
        dangerously_pull_public_prompt=True
    )

    reduce_llm = ChatOpenAI(
        model_name="gpt-4o",
        temperature=0,
        # 콜백이 모델에서 생성되는 답변 조각을 Notebook 화면에 바로 출력합니다.
        callbacks=[StreamingCallback()],
        streaming=True,
    )

    # Reduce 체인은 개별 요약을 받아 하나의 최종 문자열로 통합합니다.
    reduce_chain = reduce_prompt | reduce_llm | StrOutputParser()

    return reduce_chain.invoke(
        {
            "doc_summaries": "\n".join(doc_summaries),
            "language": "Korean"
        }
    )

In [ ]:
# [목적] 하나로 구성한 Map-Reduce 체인을 PDF 문서 목록에 실행합니다.
# docs가 Map 단계와 Reduce 단계를 차례로 거치며 최종 한국어 요약을 생성합니다.
# 완성된 결과는 answer에 저장되고 생성 과정은 설정된 콜백으로 실시간 표시됩니다.
answer = map_reduce_chain.invoke(docs)